In [2]:
import os
os.chdir("/workspaces/pythonProject/medallion_v2/data/notebooks")

# 00 · Config & helpers (Codespaces · everything in Turso)
One database does it all: **Turso** holds the source tables, the Bronze / Silver / Gold layers and the control tables. There is no local database, no Parquet landing zone and no file to lock. Every transformation is plain SQL that runs *inside* Turso, so no data is pulled into the notebook.

| Layer | Turso tables | Content |
|---|---|---|
| Source | `customers`, `sales`, ... | the operational tables (mock data) |
| Bronze | `bronze_<entity>` | everything as text + lineage (`_batch_id`, `_ingest_ts`, `_source`) |
| Silver | `silver_<entity>` | typed, cleaned, de-duplicated; `silver_quarantine`, `silver_dq_results` |
| Gold | `gold_dim_*`, `gold_fact_*` | star schema with surrogate keys and the Unknown member (-1) |
| Control | `ctl_watermark`, `ctl_batch_log` | incremental positions and run log |

Run this file with `%run -i ./00_config.ipynb` from the other notebooks. Set the Codespaces secrets `TURSO_DATABASE_URL` and `TURSO_AUTH_TOKEN` first.

In [3]:
import os, time, atexit
from pathlib import Path
import pandas as pd, yaml, libsql

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT          # works from the repo root or from notebooks/

LOAD_MODE = os.getenv("LOAD_MODE", "incremental")   # Gold load: 'incremental' (default) or 'full'
TURSO_URL, TURSO_TOKEN = os.getenv("TURSO_URL"), os.getenv("TURSO_TOKEN")

## Connection to Turso
`connect()` opens the one Turso connection (`con`). Habits that keep it trouble-free:
* a re-run cell closes the previous connection first; the connection is also closed when the kernel exits and by `release_all()` (last cell of every notebook);
* a cell that fails **rolls back** its open transaction, so nothing half-written is left behind;
* statements retry automatically on transient `busy` / `locked` / network errors.

`con.execute(sql, params)` runs a statement, `con.rows(...)` / `con.one(...)` / `con.scalar(...)` read, `con.df(...)` returns a DataFrame. Each helper commits when it finishes.

In [4]:
TRANSIENT = ("locked", "busy", "timed out", "timeout", "temporarily")
NETWORK   = ("connection", "stream", "reset", "unavailable", "502", "503", "504")

class DB:
    """Thin wrapper around the libsql connection: retries transient errors, small read helpers."""
    def __init__(self, raw): self.raw = raw
    def execute(self, sql, params=()):
        for attempt in range(5):
            try:
                return self.raw.execute(sql, tuple(params))
            except Exception as e:
                msg = str(e).lower()
                retry = any(t in msg for t in TRANSIENT) or (sql.lstrip()[:6].lower() in ("select", "pragma") and any(t in msg for t in NETWORK))
                if attempt == 4 or not retry: raise
                time.sleep(0.5 * 2 ** attempt)
    def rows(self, sql, params=()):   return self.execute(sql, params).fetchall()
    def one(self, sql, params=()):    r = self.rows(sql, params); return r[0] if r else None
    def scalar(self, sql, params=()): r = self.one(sql, params);  return r[0] if r else None
    def df(self, sql, params=()):
        cur = self.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    def commit(self):   self.raw.commit()
    def rollback(self): self.raw.rollback()
    def close(self):    self.raw.close()

con, _OPEN = None, []

def release_all():
    """Roll back and close the connection (safe to call twice)."""
    global con
    for h in _OPEN:
        for f in (h.rollback, h.close):
            try: f()
            except Exception: pass
    _OPEN.clear(); con = None
atexit.register(release_all)

def connect():
    """Open the Turso connection (any earlier one from this kernel is closed first)."""
    global con
    release_all()
    test_db = os.getenv("TEST_DB")                      # testing hook only: a libSQL file standing in for Turso
    if test_db:
        raw = libsql.connect(test_db)
    elif TURSO_URL and TURSO_TOKEN:
        raw = libsql.connect(database=TURSO_URL, auth_token=TURSO_TOKEN)
    else:
        raise RuntimeError("Set the Codespaces secrets TURSO_DATABASE_URL and TURSO_AUTH_TOKEN, then restart the kernel.")
    con = DB(raw); _OPEN.append(con)
    return con

def _on_cell_done(result):                              # a failed cell must not leave a transaction open
    if getattr(result, "error_in_exec", None) and con is not None:
        try: con.rollback()
        except Exception: pass
try:
    _ip = get_ipython()
    if "_cb" in globals():
        try: _ip.events.unregister("post_run_cell", _cb)
        except Exception: pass
    _cb = _on_cell_done; _ip.events.register("post_run_cell", _cb)
except Exception:
    pass

## Control tables & run batch
`ctl_watermark` holds every incremental position (source extract, Silver and Gold per object), `ctl_batch_log` numbers the runs, `silver_quarantine` keeps rejected rows with the rules they failed, `silver_dq_results` records every rule's failed / checked counts per run.
`reset_domain(...)` drops everything a domain created (for a clean start).

In [5]:
BATCH_ID = None

def open_warehouse():
    """Connect and make sure the control tables exist."""
    connect()
    con.execute("CREATE TABLE IF NOT EXISTS ctl_watermark (name TEXT PRIMARY KEY, value TEXT, updated_ts TEXT)")
    con.execute("CREATE TABLE IF NOT EXISTS ctl_batch_log (batch_id INTEGER PRIMARY KEY, started_ts TEXT, load_mode TEXT)")
    con.execute("CREATE TABLE IF NOT EXISTS silver_quarantine (entity TEXT, batch_id INTEGER, payload TEXT, _ts TEXT, _errors TEXT)")
    con.execute("""CREATE TABLE IF NOT EXISTS silver_dq_results (entity TEXT, batch_id INTEGER, rule TEXT,
                   criticality TEXT, function TEXT, failed_rows INTEGER, checked_rows INTEGER, _ts TEXT)""")
    con.commit()
    return con

def get_wm(name):
    return con.scalar("SELECT value FROM ctl_watermark WHERE name = ?", [name])

def set_wm(name, value):
    con.execute("""INSERT INTO ctl_watermark VALUES (?, ?, datetime('now'))
                   ON CONFLICT (name) DO UPDATE SET value = excluded.value, updated_ts = excluded.updated_ts""", [name, str(value)])

def new_batch():
    global BATCH_ID
    BATCH_ID = con.scalar("SELECT coalesce(max(batch_id), 0) + 1 FROM ctl_batch_log")
    con.execute("INSERT INTO ctl_batch_log VALUES (?, datetime('now'), ?)", [BATCH_ID, LOAD_MODE])
    con.commit()
    return BATCH_ID

def reset_domain(dom):
    """Start over: drop a domain's source, Bronze, Silver and Gold tables and clear its watermarks and rejects."""
    ents, objs = dom["entities"], dom["unified"] + dom["dims"] + dom["facts"]
    for t in [*ents, *[f"bronze_{e}" for e in ents], *[f"silver_{e}" for e in ents + dom["unified"]],
              *[f"gold_{x}" for x in dom["dims"] + dom["facts"]]]:
        con.execute(f"DROP TABLE IF EXISTS {t}")
    names = [f"{p}.{x}" for x in ents for p in ("extract", "silver")] + [f"silver.{u}" for u in dom["unified"]] + [f"gold.{x}" for x in dom["dims"] + dom["facts"]]
    con.execute(f"DELETE FROM ctl_watermark WHERE name IN ({','.join('?' * len(names))})", names)
    for t in ("silver_quarantine", "silver_dq_results"):
        con.execute(f"DELETE FROM {t} WHERE entity IN ({','.join('?' * len(ents))})", ents)
    con.commit()

def upsert_csv(table, path, key, chunk=200):
    """Mock-source loader: upsert a CSV into a source table, `chunk` rows per statement (blank cell -> NULL)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    cols = list(df.columns)
    rows = [tuple(None if v == "" else v for v in r) for r in df.itertuples(index=False)]
    upd = ", ".join(f"{c} = excluded.{c}" for c in cols if c != key)
    one = "(" + ", ".join("?" * len(cols)) + ")"
    t = time.time()
    for i in range(0, len(rows), chunk):
        part = rows[i:i + chunk]
        con.execute(f"INSERT INTO {table} ({', '.join(cols)}) VALUES {', '.join([one] * len(part))} "
                    f"ON CONFLICT({key}) DO UPDATE SET {upd}", [v for r in part for v in r])
    con.commit()
    return len(rows), time.time() - t

## Domain definitions (YAML): tables, columns, types, keys, dimensions, facts
`load_domain("sales")` reads `domains/sales.yml`, validates it (unknown column, missing key, bad lookup... all reported at once) and registers its objects in four dictionaries used by every helper below:

| Dictionary | Holds |
|---|---|
| `ENTITIES` | Silver tables: keys, `order_by`, source label and `(name, logical type[, transform])` columns |
| `DIMS` | Gold dimensions: source, business key, surrogate key, attributes (plain or derived) |
| `FACTS` | Gold facts: source entity, grain, extra joins, columns (plain, expression or dimension lookup) |
| `UNIFIED` | Combined Silver tables built from several sources (used by `customer360`) |

Logical types in the YAML map to Turso storage: `VARCHAR`→TEXT, `INTEGER`/`BIGINT`→INTEGER, `DECIMAL(p,s)`/`DOUBLE`→REAL (rounded to `s`), `DATE`/`TIMESTAMP`→ISO-8601 TEXT, `BOOLEAN`→INTEGER.

In [6]:
ENTITIES, DIMS, FACTS, UNIFIED = {}, {}, {}, {}      # filled by load_domain(); empty until a notebook asks for a domain

def find_file(name):
    for d in (Path.cwd(), ROOT, ROOT / "domains", ROOT / "rules", ROOT / "data", ROOT / "sql"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found next to the notebook or in domains/ or rules/")

def _check_domain(d):
    """Collect every structural problem in a domain file so they can be fixed in one pass."""
    errs, ents, unified = [], d.get("entities") or {}, d.get("unified") or {}
    cols_of = {e: [c.get("name") for c in s.get("columns", [])] for e, s in ents.items()}
    for e, s in ents.items():
        for c in s.get("columns", []):
            if "name" not in c or "type" not in c: errs.append(f"entity {e}: every column needs name and type ({c})")
        for k in list(s.get("keys", [])) + [s.get("order_by")]:
            if k not in cols_of[e]: errs.append(f"entity {e}: key/order_by '{k}' is not one of its columns")
    for u, s in unified.items():
        for a, m in s.get("sources", {}).items():
            if m["entity"] not in ents: errs.append(f"unified {u}: source {a} uses unknown entity {m['entity']}")
        for attr in s.get("attributes", []):
            for ref in attr["from"]:
                a, _, c = ref.partition(".")
                m = s["sources"].get(a)
                if not m or c not in cols_of.get(m["entity"], []): errs.append(f"unified {u}: '{ref}' is not a source column")
        for flag, a in s.get("flags", {}).items():
            if a not in s["sources"]: errs.append(f"unified {u}: flag {flag} uses unknown source {a}")
    tables = {**{e: set(c) for e, c in cols_of.items()},
              **{u: {s["key"], s["match_on"], *[a["name"] for a in s.get("attributes", [])], *s.get("flags", {}), "source_count"}
                 for u, s in unified.items()}}
    dims = d.get("dimensions") or {}
    for n, s in dims.items():
        if s["source"] not in tables: errs.append(f"dimension {n}: unknown source {s['source']}"); continue
        for a in s["attributes"]:
            if isinstance(a, str) and a not in tables[s["source"]]: errs.append(f"dimension {n}: attribute '{a}' is not a column of {s['source']}")
        if s["key"] not in tables[s["source"]]: errs.append(f"dimension {n}: key '{s['key']}' is not a column of {s['source']}")
    for n, f in (d.get("facts") or {}).items():
        if f["source"] not in ents: errs.append(f"fact {n}: unknown source entity {f['source']}"); continue
        names = [c["name"] for c in f["columns"]]
        for g in f["grain"]:
            if g not in names: errs.append(f"fact {n}: grain column '{g}' is not a fact column")
        for j in f.get("joins", []):
            if j["entity"] not in ents: errs.append(f"fact {n}: join to unknown entity {j['entity']}")
        for c in f["columns"]:
            if "lookup" in c:
                if c["lookup"]["dim"] not in dims and c["lookup"]["dim"] not in DIMS: errs.append(f"fact {n}: column {c['name']} looks up unknown dimension {c['lookup']['dim']}")
            elif "expr" not in c and c["name"] not in cols_of[f["source"]]:
                errs.append(f"fact {n}: column '{c['name']}' is not in {f['source']} (add an expr or a lookup)")
    return errs

def load_domain(name):
    """Read domains/<name>.yml, validate it, register its objects, and return the names it defines."""
    d = yaml.safe_load(find_file(f"{name}.yml").read_text())
    errs = _check_domain(d)
    if errs: raise ValueError(f"Invalid domain '{name}':\n  " + "\n  ".join(errs))
    for e, s in d["entities"].items():
        ENTITIES[e] = dict(keys=list(s["keys"]), order_by=s["order_by"], landing=s.get("landing", e),
                           columns=[(c["name"], c["type"], c["transform"]) if c.get("transform") else (c["name"], c["type"]) for c in s["columns"]])
    for u, s in (d.get("unified") or {}).items():
        attrs = [(a["name"], a["type"], a["from"]) for a in s["attributes"]]
        UNIFIED[u] = dict(key=s["key"], match_on=s["match_on"], sources={a: (m["entity"], m["key"]) for a, m in s["sources"].items()},
                          attributes=attrs, flags=list(s.get("flags", {}).items()),
                          columns=[(s["key"], "VARCHAR"), (s["match_on"], "VARCHAR"), *[(n, t) for n, t, _ in attrs],
                                   *[(f, "BOOLEAN") for f in s.get("flags", {})], ("source_count", "INTEGER")])
    for n, s in (d.get("dimensions") or {}).items():
        DIMS[n] = dict(src=s["source"], key=s["key"], sk=s["sk"], attrs=s["attributes"])
    for n, s in (d.get("facts") or {}).items():
        FACTS[n] = s
    out = dict(domain=name, entities=list(d["entities"]), dims=list(d.get("dimensions") or {}),
               facts=list(d.get("facts") or {}), unified=list(d.get("unified") or {}))
    print(f"domain '{name}': {len(out['entities'])} entities, {len(out['dims'])} dimensions, {len(out['facts'])} facts")
    return out

# ---- logical YAML types -> Turso storage, and the text transforms the YAML can name ----
def sqltype(t):
    t = t.upper()
    if t.startswith(("INT", "BIGINT", "BOOL")): return "INTEGER"
    if t.startswith(("DECIMAL", "DOUBLE", "REAL", "FLOAT", "NUMERIC")): return "REAL"
    return "TEXT"                                    # VARCHAR, TEXT, DATE, TIMESTAMP (ISO-8601 text)

_PHONE = "{}"
for _ch in (" ", "-", "(", ")", ".", "+"): _PHONE = f"replace({_PHONE}, '{_ch}', '')"
TRANSFORMS = {"lower": "lower({})", "upper": "upper({})", "digits": f"substr({_PHONE}, -10)"}   # digits = last 10 digits

## Data-quality rule library (DQX YAML)
Rules live in **`dq_rules.yml`**, one list of [DQX](https://databrickslabs.github.io/dqx/) checks per entity (`criticality`, `name`, `check: {function, arguments}`, optional `filter` and `for_each_column`).

* `DQEngine.validate_checks` from the DQX module validates every rule when the file is loaded (unknown function, wrong argument type, missing key).
* **Execution:** the same rules are translated to SQL (`compile_rules`) and run inside Turso, so no Spark session is needed. `error` rules quarantine the row; `warn` rules keep it and are counted in `silver_dq_results`.
* Turso has no regular-expression engine, so `regex_match` supports the patterns registered in `REGEX_SQL` (the e-mail format is included); anything else should be written as a `sql_expression` with `LIKE` / `GLOB`.
* `native_dqx_check(entity)` (optional) runs the **identical YAML natively on DQX/Spark** and compares the counts, which proves the two executors agree.

In [7]:
import sys, subprocess, importlib, re
try:
    from databricks.labs.dqx.engine import DQEngine, DQEngineCore
except ImportError:                                   # first run: install the DQX module into this kernel
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "databricks-labs-dqx"])
    importlib.invalidate_caches()
    from databricks.labs.dqx.engine import DQEngine, DQEngineCore

def load_rules(path=None):
    """Read the YAML rule library and validate every entity's checks with DQX itself."""
    path = path or find_file("dq_rules.yml")
    raw = yaml.safe_load(path.read_text()) or {}
    bad = {}
    for entity, checks in raw.items():
        status = DQEngine.validate_checks(checks or [])
        if status.has_errors: bad[entity] = status.errors
    if bad: raise ValueError(f"Invalid DQX rules in {path.name}: {bad}")
    return raw

RULES = load_rules()
print(f"rule library: {sum(len(v) for v in RULES.values())} checks for {len(RULES)} entities")

rule library: 27 checks for 9 entities


In [8]:
# ---- SQL executor for DQX checks: each supported function becomes a "violation" predicate (TRUE = rule failed).
def _lit(v):
    """DQX convention: quoted text = string literal, number = number, anything else = column/SQL."""
    if isinstance(v, bool): return "1" if v else "0"
    if isinstance(v, (int, float)): return repr(v)
    return str(v).strip()                              # dates are ISO text in Turso, so '2026-01-01' compares correctly

# Turso/SQLite has no regex engine: the regex patterns used in dq_rules.yml are mapped to an equivalent LIKE/GLOB test.
REGEX_SQL = {
    r"^[^@\s]+@[^@\s]+\.[^@\s]+$":                     # e-mail format
        "({c} GLOB '?*@?*.?*' AND {c} NOT GLOB '* *' AND length({c}) - length(replace({c}, '@', '')) = 1 "
        "AND substr({c}, instr({c}, '@') + 1) GLOB '?*.?*')",
}

def _violation(fn, a):
    c = f'"{a["column"]}"' if a.get("column") else None
    ci = a.get("case_sensitive", True)
    def values(key):
        return ", ".join((f"lower({_lit(x)})" if not ci else _lit(x)) for x in a[key])
    lhs = f"lower(CAST({c} AS TEXT))" if (c and not ci) else c
    if fn == "is_not_null":                return f"{c} IS NULL"
    if fn == "is_null":                    return f"{c} IS NOT NULL"
    if fn == "is_not_null_and_not_empty":
        v = f"trim(CAST({c} AS TEXT))" if a.get("trim_strings") else f"CAST({c} AS TEXT)"
        return f"({c} IS NULL OR {v} = '')"
    if fn == "is_in_list":                 return f"({c} IS NOT NULL AND {lhs} NOT IN ({values('allowed')}))"
    if fn == "is_not_null_and_is_in_list": return f"({c} IS NULL OR {lhs} NOT IN ({values('allowed')}))"
    if fn == "is_not_in_list":             return f"{lhs} IN ({values('forbidden')})"
    if fn == "is_not_less_than":           return f"{c} < {_lit(a['limit'])}"
    if fn == "is_not_greater_than":        return f"{c} > {_lit(a['limit'])}"
    if fn == "is_in_range":                return f"({c} < {_lit(a['min_limit'])} OR {c} > {_lit(a['max_limit'])})"
    if fn == "is_not_in_range":            return f"({c} >= {_lit(a['min_limit'])} AND {c} <= {_lit(a['max_limit'])})"
    if fn == "regex_match":
        if a["regex"] not in REGEX_SQL:
            raise NotImplementedError(f"regex {a['regex']!r} has no SQL translation (Turso has no regex engine). "
                                      f"Add it to REGEX_SQL, or write the rule as a sql_expression with LIKE / GLOB.")
        ok = REGEX_SQL[a["regex"]].format(c=c)
        return ok if a.get("negate") else f"(NOT {ok})"
    if fn == "sql_expression":             return f"({a['expression']})" if a.get("negate") else f"(NOT ({a['expression']}))"
    raise NotImplementedError(f"DQX function '{fn}' has no SQL translation yet; supported: {sorted(SUPPORTED)}. "
                              f"Add it to _violation(), or run it natively with native_dqx_check().")

SUPPORTED = {"is_unique", "is_not_null", "is_null", "is_not_null_and_not_empty", "is_in_list", "is_not_null_and_is_in_list", "is_not_in_list",
             "is_not_less_than", "is_not_greater_than", "is_in_range", "is_not_in_range", "regex_match", "sql_expression"}

def compile_rules(entity):
    """Expand for_each_column, apply filters, and return [{name, criticality, function, sql}] for one entity.
    `is_unique` (de-duplication on several columns) is dataset-level: its rule SQL reads helper columns (_dupN, _dkN)
    that clean_to_silver adds; `unique` carries the columns."""
    out = []
    for chk in RULES.get(entity, []):
        spec = chk["check"]; fn = spec["function"]; args = dict(spec.get("arguments") or {})
        if fn == "is_unique":
            n = sum(1 for r in out if "unique" in r)
            out.append(dict(name=chk.get("name") or "_".join(args["columns"]) + "_is_unique", criticality=chk.get("criticality", "error"),
                            function=fn, sql=f"(coalesce(_dup{n} > 1, 0) OR coalesce(_dk{n}, 0) = 1)",
                            unique=dict(idx=n, columns=list(args["columns"]))))
            continue
        each = spec.get("for_each_column")
        for a in ([dict(args, column=col) for col in each] if each else [args]):
            col = a.get("column")
            name = chk.get("name") or (f"{col}_{fn}" if col else fn)
            if each and chk.get("name"): name = f"{name}_{col}"
            v = _violation(fn, a)
            if chk.get("filter"): v = f"(({chk['filter']}) AND {v})"
            out.append(dict(name=name, criticality=chk.get("criticality", "error"), function=fn, sql=f"coalesce(({v}), 0)"))
    return out

def _name_list(rules):
    """SQL text holding the comma-separated names of the rules a row fails ('' = none)."""
    if not rules: return "''"
    parts = " || ".join("coalesce(CASE WHEN {} THEN '{},' END, '')".format(r["sql"], r["name"].replace("'", "''")) for r in rules)
    return f"trim({parts}, ',')"

## Step 1 helper · Stage: source tables → Bronze
Copies only rows with `updated_at >= last watermark` from the source table into `bronze_<entity>` with one SQL statement inside Turso: everything as text plus lineage columns (`_batch_id`, `_ingest_ts`, `_source`). Rows already in Bronze (same key and `updated_at`) are skipped, so re-runs never duplicate. New source columns are added to Bronze automatically.

In [9]:
def _cols(table):
    return [r[1] for r in con.rows(f"PRAGMA table_info({table})")]

def stage(entity):
    e, key = ENTITIES[entity], ENTITIES[entity]["keys"][0]
    wm = get_wm(f"extract.{entity}")                              # ISO timestamps compare correctly as text
    cols = _cols(entity)
    if not cols: raise RuntimeError(f"source table '{entity}' does not exist: run the load notebook (01 or 03) first")
    where, args = ("s.updated_at >= ?", [wm]) if wm else ("1 = 1", [])
    n, max_ts = con.one(f"SELECT count(*), max(s.updated_at) FROM {entity} s WHERE {where}", args)
    if n == 0:
        print(f"  {entity:18s} nothing new in source"); return
    con.execute(f"""CREATE TABLE IF NOT EXISTS bronze_{entity}
                    ({', '.join(f'"{c}" TEXT' for c in cols)}, _batch_id INTEGER, _ingest_ts TEXT, _source TEXT)""")
    have = set(_cols(f"bronze_{entity}"))
    for c in cols:
        if c not in have: con.execute(f'ALTER TABLE bronze_{entity} ADD COLUMN "{c}" TEXT')    # schema evolution
    names = ", ".join(f'"{c}"' for c in cols)
    cast  = ", ".join(f'CAST(s."{c}" AS TEXT)' for c in cols)
    con.execute(f"""INSERT INTO bronze_{entity} ({names}, _batch_id, _ingest_ts, _source)
                    SELECT {cast}, {BATCH_ID}, datetime('now'), '{e['landing']}' FROM {entity} s
                    WHERE {where} AND NOT EXISTS (SELECT 1 FROM bronze_{entity} b
                          WHERE b."{key}" = CAST(s."{key}" AS TEXT) AND b.updated_at = s.updated_at)""", args)
    new = con.scalar(f"SELECT count(*) FROM bronze_{entity} WHERE _batch_id = {BATCH_ID}")
    set_wm(f"extract.{entity}", max_ts)                           # only after Bronze succeeded
    con.commit()
    print(f"  {entity:18s} extracted {n:5d} rows (since {wm or 'the beginning'}), {new:5d} new in bronze")

## Step 2 helpers · Clean: Bronze → Silver
Blank text becomes NULL, values are cast to the YAML types (a value that cannot be cast becomes NULL), then trim / lower / upper / digits, then the **DQX rules from `dq_rules.yml`**: rows failing an `error` rule go to `silver_quarantine` (raw row as JSON plus `_errors`, the names of the failed rules); `warn` rules keep the row and are only counted. Every rule's failed/checked counts per run go to `silver_dq_results`. Then latest record per key and a guarded upsert: only rows whose values actually changed are updated, and those get `_silver_batch = this run`; that column is what makes the Gold load incremental.

In [10]:
def merge(target, source, keys, cols, batch_col, guard="1", sk=None):
    """Generic upsert (INSERT ... ON CONFLICT): insert new keys, update only when a value changed (and the guard passes).
    Returns how many rows this run inserted or updated."""
    attrs = [c for c in cols if c not in keys]
    diff  = " OR ".join(f"{target}.{c} IS NOT excluded.{c}" for c in attrs)
    sets  = ", ".join(f"{c} = excluded.{c}" for c in attrs) + f", {batch_col} = {BATCH_ID}"
    icols = ([sk[0]] if sk else []) + cols + [batch_col]
    ivals = ([sk[1]] if sk else []) + [f"s.{c}" for c in cols] + [str(BATCH_ID)]
    con.execute(f"""INSERT INTO {target} ({', '.join(icols)})
                    SELECT {', '.join(ivals)} FROM {source} AS s WHERE true
                    ON CONFLICT ({', '.join(keys)}) DO UPDATE SET {sets}
                    WHERE ({guard}) AND ({diff})""")
    return con.scalar(f"SELECT count(*) FROM {target} WHERE {batch_col} = {BATCH_ID}")

_J = "'[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]*'"             # ISO date prefix

def _expr(col):
    """SQL that turns a cleaned text value `"name"` into its YAML type; a value that does not fit becomes NULL."""
    name, typ, *fn = col
    v, t = f'"{name}"', typ.upper()
    if t.startswith(("VARCHAR", "TEXT")):
        return TRANSFORMS[fn[0]].format(v) if fn else v
    if t.startswith(("INT", "BIGINT")):
        return f"CASE WHEN json_valid({v}) THEN CASE WHEN json_type({v}) = 'integer' THEN CAST({v} AS INTEGER) END END"
    if t.startswith(("DECIMAL", "DOUBLE", "REAL", "FLOAT")):
        s = re.search(r"DECIMAL\(\d+,\s*(\d+)\)", t)
        num = f"CAST({v} AS REAL)"
        return f"CASE WHEN json_valid({v}) THEN CASE WHEN json_type({v}) IN ('integer', 'real') THEN {f'round({num}, {s.group(1)})' if s else num} END END"
    if t == "DATE":      return f"CASE WHEN {v} GLOB {_J} THEN date({v}) END"
    if t == "TIMESTAMP": return f"CASE WHEN {v} GLOB {_J} THEN strftime('%Y-%m-%d %H:%M:%S', {v}) END"
    raise NotImplementedError(f"type {typ} in the domain YAML")

def _typed_sql(entity, where):
    """Bronze rows -> trimmed, blank-to-NULL, typed columns (+ the Bronze rowid and ingest time)."""
    cols = ENTITIES[entity]["columns"]
    inner = ", ".join(f"nullif(trim(\"{c[0]}\"), '') AS \"{c[0]}\"" for c in cols)
    outer = ", ".join(f'{_expr(c)} AS "{c[0]}"' for c in cols)
    return f"SELECT {outer}, _rid, _ingest_ts FROM (SELECT {inner}, rowid AS _rid, _ingest_ts FROM bronze_{entity} WHERE {where})"

def _work(name, select):
    con.execute(f"DROP TABLE IF EXISTS {name}")
    con.execute(f"CREATE TABLE {name} AS {select}")

def clean_to_silver(entity):
    c, cols = ENTITIES[entity], ENTITIES[entity]["columns"]
    names, keys = [x[0] for x in cols], ENTITIES[entity]["keys"]
    last = int(get_wm(f"silver.{entity}") or 0)
    top = con.scalar(f"SELECT coalesce(max(_batch_id), 0) FROM bronze_{entity}") if _cols(f"bronze_{entity}") else 0
    if top <= last:
        print(f"  {entity:18s} nothing new in bronze"); return
    rules = compile_rules(entity)                                   # rules come from dq_rules.yml (DQX format)
    errs  = [r for r in rules if r["criticality"] == "error"]
    warns = [r for r in rules if r["criticality"] != "error"]
    con.execute(f"""CREATE TABLE IF NOT EXISTS silver_{entity}
        ({', '.join(f'{x[0]} {sqltype(x[1])}' for x in cols)}, _silver_batch INTEGER, PRIMARY KEY ({', '.join(keys)}))""")
    # de-duplication rules (is_unique): the FIRST submitted record of a group wins, later copies are rejected.
    # _dupN = position inside this batch's group (oldest first); _dkN = 1 when a different record with the same values is already in Silver
    extra = ""
    for u in (r["unique"] for r in rules if "unique" in r):
        cs, n = u["columns"], u["idx"]
        notnull = " AND ".join(f'"{x}" IS NOT NULL' for x in cs)                 # NULLs count as distinct (DQX default)
        same    = " AND ".join(f'o."{x}" = t."{x}"' for x in cs)
        other   = " OR ".join(f'o."{k}" <> t."{k}"' for k in keys)
        extra += (f', CASE WHEN {notnull} THEN row_number() OVER (PARTITION BY {", ".join(chr(34) + x + chr(34) for x in cs)} '
                  f'ORDER BY "{c["order_by"]}", _rid) END AS _dup{n}'
                  f', CASE WHEN {notnull} AND EXISTS (SELECT 1 FROM silver_{entity} o WHERE {same} AND ({other})) THEN 1 ELSE 0 END AS _dk{n}')
    _work("wrk_typed", f"SELECT t.*{extra} FROM ({_typed_sql(entity, f'_batch_id > {last}')}) t")
    _work("wrk_flagged", f"SELECT *, {_name_list(errs)} AS _errors, {_name_list(warns)} AS _warnings FROM wrk_typed")
    if rules:                                                       # rule-level metrics (like DQX's summary metrics)
        sums = ", ".join("coalesce(sum(CASE WHEN {} THEN 1 ELSE 0 END), 0)".format(r["sql"]) for r in rules)
        counts = con.one(f"SELECT count(*), {sums} FROM wrk_typed")
        con.execute("INSERT INTO silver_dq_results VALUES " + ", ".join(["(?, ?, ?, ?, ?, ?, ?, datetime('now'))"] * len(rules)),
                    [v for k, r in enumerate(rules) for v in (entity, BATCH_ID, r["name"], r["criticality"], r["function"], counts[k + 1], counts[0])])
    bcols = _cols(f"bronze_{entity}")
    payload = "json_object(" + ", ".join(f"'{b}', b.\"{b}\"" for b in bcols) + ")"
    con.execute(f"""INSERT INTO silver_quarantine (entity, batch_id, payload, _ts, _errors)
        SELECT '{entity}', {BATCH_ID}, {payload}, datetime('now'), f._errors FROM bronze_{entity} b
        JOIN wrk_flagged f ON b.rowid = f._rid WHERE f._errors <> ''""")
    bad = con.scalar("SELECT count(*) FROM wrk_flagged WHERE _errors <> ''")
    _work("wrk_good", f"""SELECT {', '.join(names)} FROM (
        SELECT *, row_number() OVER (PARTITION BY {', '.join(keys)} ORDER BY {c['order_by']} DESC, _ingest_ts DESC, _rid DESC) AS _rn
        FROM wrk_flagged WHERE _errors = '') WHERE _rn = 1""")
    ob, tgt = c["order_by"], f"silver_{entity}"
    changed = merge(tgt, "wrk_good", keys, names, "_silver_batch", guard=f"excluded.{ob} >= {tgt}.{ob}")
    set_wm(f"silver.{entity}", top)
    good   = con.scalar("SELECT count(*) FROM wrk_good")
    warned = con.scalar("SELECT count(*) FROM wrk_flagged WHERE _errors = '' AND _warnings <> ''")
    con.commit()
    print(f"  {entity:18s} {good:5d} valid latest rows -> {changed:5d} inserted/updated, {bad:3d} quarantined, {warned:3d} with warnings")

In [11]:
def native_dqx_check(entity):
    """OPTIONAL: run the same YAML natively with DQX on a local Spark session and compare with the SQL executor.
    Needs Java + pyspark; compares, over all of Bronze for the entity, the rows failing an error rule and the rows with a warning."""
    try:
        import datetime as dt
        from unittest.mock import MagicMock
        from pyspark.sql import SparkSession, functions as F, types as T
        cols = ENTITIES[entity]["columns"]
        rules = compile_rules(entity)
        uniq  = [r for r in rules if "unique" in r]                    # dataset-level: compared separately below
        rules = [r for r in rules if "unique" not in r]
        errs  = [r for r in rules if r["criticality"] == "error"]; warns = [r for r in rules if r["criticality"] != "error"]
        anyv = lambda rs: " OR ".join(r["sql"] for r in rs) or "0"
        typed = _typed_sql(entity, "1 = 1")
        d_err, d_warn = con.one(f"SELECT coalesce(sum(CASE WHEN {anyv(errs)} THEN 1 ELSE 0 END), 0), "
                                f"coalesce(sum(CASE WHEN {anyv(warns)} THEN 1 ELSE 0 END), 0) FROM ({typed})")
        rows = con.rows("SELECT " + ", ".join(f'"{c[0]}"' for c in cols) + f" FROM ({typed})")
        kind = {"INTEGER": (T.LongType(), None), "REAL": (T.DoubleType(), None), "TEXT": (T.StringType(), None)}
        def spark_col(c):
            t = c[1].upper()
            if t == "DATE":      return T.DateType(), (lambda v: dt.date.fromisoformat(v))
            if t == "TIMESTAMP": return T.TimestampType(), (lambda v: dt.datetime.fromisoformat(v))
            return kind[sqltype(t)]
        spec = [spark_col(c) for c in cols]
        schema = T.StructType([T.StructField(c[0], s[0], True) for c, s in zip(cols, spec)])
        data = [tuple(None if v is None else (s[1](v) if s[1] else v) for v, s in zip(r, spec)) for r in rows]
        spark = (SparkSession.builder.master("local[1]").config("spark.ui.enabled", "false")
                 .config("spark.driver.bindAddress", "127.0.0.1").config("spark.driver.host", "127.0.0.1").getOrCreate())
        engine = DQEngineCore(workspace_client=MagicMock(), spark=spark)   # no Databricks workspace needed for row checks
        sdf = spark.createDataFrame(data, schema)
        row_checks = [x for x in RULES[entity] if x["check"]["function"] != "is_unique"]
        res = engine.apply_checks_by_metadata(sdf, row_checks)
        n_err  = res.filter(F.col("_errors").isNotNull()).count()
        n_warn = res.filter(F.col("_warnings").isNotNull()).count()
        ok = (n_err, n_warn) == (d_err, d_warn)
        extra = ""
        for u in (r["unique"] for r in uniq):                           # DQX flags every row of a duplicate group; SQL keeps the first one
            part = ", ".join(f'"{x}"' for x in u["columns"]); nn = " AND ".join(f'"{x}" IS NOT NULL' for x in u["columns"])
            d_grp = con.scalar(f"SELECT count(*) FROM (SELECT count(*) OVER (PARTITION BY {part}) AS n FROM ({typed}) WHERE {nn}) WHERE n > 1")
            n_grp = engine.apply_checks_by_metadata(sdf, [x for x in RULES[entity] if x["check"]["function"] == "is_unique"]) \
                          .filter(F.col("_errors").isNotNull()).count()
            ok = ok and d_grp == n_grp
            extra = f" | rows in duplicate groups SQL/DQX = {d_grp}/{n_grp}"
        print(f"  {entity:18s} SQL errors/warnings = {d_err}/{d_warn} | native DQX = {n_err}/{n_warn}{extra} -> {'MATCH' if ok else 'MISMATCH'}")
        return ok
    except Exception as e:
        print(f"  {entity:18s} native DQX check skipped ({type(e).__name__}: {str(e)[:120]})")

## Steps 3–5 helpers · Join, merge to Gold, and the incremental delta
Dimensions and facts are processed the same way: **one generic function per step, driven by the domain YAML**.

| Step | Dimension | Fact |
|---|---|---|
| Join | `load_dim(dim)` (join + merge in one go) | `build_fact(fact)`: source + joins + dimension lookups into a work table |
| Merge | (inside `load_dim`) | `merge_fact(fact)`: guarded upsert into `gold_<fact>` and move its watermark |

Both use `delta_predicate`, so only Silver rows changed since the last Gold load are processed (`LOAD_MODE=full` takes everything). A fact row whose dimension key was unknown (-1) is **re-driven** on later runs until the dimension arrives.

In [12]:
PENDING = {}

def delta_predicate(target, silver_table):
    wm  = get_wm(f"gold.{target}")
    end = con.scalar(f"SELECT coalesce(max(_silver_batch), 0) FROM silver_{silver_table}")
    PENDING[target] = end
    if LOAD_MODE == "full" or wm is None:
        return "1 = 1"
    return f"s._silver_batch > {int(wm)} AND s._silver_batch <= {end}"

def _types(table):
    """Logical column types of a Silver entity or a unified table, from the domain definitions."""
    return {c[0]: c[1] for c in ENTITIES[table]["columns"]} if table in ENTITIES else dict(UNIFIED[table]["columns"])

def load_dim(dim):
    d = DIMS[dim]; tgt, sk, key = f"gold_{dim}", d["sk"], d["key"]
    types = _types(d["src"])
    attrs = [(a, types[a], a) if isinstance(a, str) else (a["name"], a["type"], a["expr"]) for a in d["attrs"]]   # (name, type, select expr)
    unknown = "'-1'" if sqltype(types[key]) == "TEXT" else "-1"
    con.execute(f"""CREATE TABLE IF NOT EXISTS {tgt} ({sk} INTEGER PRIMARY KEY, {key} {sqltype(types[key])} NOT NULL UNIQUE,
                    {', '.join(f'{n} {sqltype(t)}' for n, t, _ in attrs)}, _gold_batch INTEGER)""")
    con.execute(f"INSERT INTO {tgt} ({sk}, {key}) SELECT -1, {unknown} WHERE NOT EXISTS (SELECT 1 FROM {tgt} WHERE {sk} = -1)")
    pred = delta_predicate(dim, d["src"])
    cols = [key, *[n for n, _, _ in attrs]]
    sel  = ", ".join([key, *[(n if e == n else f"{e} AS {n}") for n, _, e in attrs]])
    new_sk = (f"max((SELECT max({sk}) FROM {tgt}), 0) + sum(CASE WHEN EXISTS (SELECT 1 FROM {tgt} t0 WHERE t0.{key} = s.{key}) "
              f"THEN 0 ELSE 1 END) OVER (ORDER BY s.{key})")                  # next free number per NEW key: 1, 2, 3 ... (-1 = Unknown member)
    changed = merge(tgt, f"(SELECT {sel} FROM silver_{d['src']} s WHERE {pred})", [key], cols, "_gold_batch", sk=(sk, new_sk))
    set_wm(f"gold.{dim}", PENDING[dim]); con.commit()
    print(f"  {dim:16s} {changed:5d} inserted/updated")

def build_fact(fact):
    """Step 3 for one fact: join its source to the other Silver tables and the dimensions; returns the rows ready to merge."""
    f, tgt = FACTS[fact], f"gold_{fact}"
    types = _types(f["source"])
    cols  = [(c["name"], c.get("type") or types[c["name"]]) for c in f["columns"]]
    con.execute(f"""CREATE TABLE IF NOT EXISTS {tgt} ({', '.join(f'{n} {sqltype(t)}' for n, t in cols)},
                    _gold_batch INTEGER, PRIMARY KEY ({', '.join(f['grain'])}))""")
    pred = delta_predicate(fact, f["source"])
    joins = [f"LEFT JOIN silver_{j['entity']} {j['alias']} ON {j['condition']}" for j in f.get("joins", [])]
    sel, unknown_sk = [], []
    for k, c in enumerate(f["columns"]):
        if "lookup" in c:                                               # surrogate key from a dimension, -1 = Unknown member
            dm = DIMS[c["lookup"]["dim"]]; a = f"d{k}"
            joins.append(f"LEFT JOIN gold_{c['lookup']['dim']} {a} ON {a}.{dm['key']} = {c['lookup']['by']}")
            sel.append(f"coalesce({a}.{dm['sk']}, -1) AS {c['name']}"); unknown_sk.append(c["name"])
        elif "expr" in c:
            sel.append(f"{c['expr']} AS {c['name']}")
        else:
            sel.append(f"s.{c['name']} AS {c['name']}")
    late = ""
    if pred != "1 = 1" and unknown_sk:                                  # incremental: also re-drive rows with an unknown dimension key
        g = ", ".join(f["grain"]); gs = ", ".join(f"s.{x}" for x in f["grain"])
        late = f" OR ({gs}) IN (SELECT {g} FROM {tgt} WHERE {' OR '.join(f'{x} = -1' for x in unknown_sk)})"
    _work(f"wrk_fact_{fact}", f"""SELECT {', '.join(sel)} FROM silver_{f['source']} s {' '.join(joins)} WHERE ({pred}){late}""")
    return con.scalar(f"SELECT count(*) FROM wrk_fact_{fact}")

def merge_fact(fact):
    """Step 4 for one fact: guarded upsert into Gold, then move the fact's watermark."""
    f = FACTS[fact]
    names = [c["name"] for c in f["columns"]]
    changed = merge(f"gold_{fact}", f"wrk_fact_{fact}", f["grain"], names, "_gold_batch")
    set_wm(f"gold.{fact}", PENDING[fact]); con.commit()
    return changed